In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()
customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

sales = sales.withColumn("sale_date", to_date(col("sale_date")))

df = sales.join(customers,"customer_id").join(cars,"car_id")
df.groupBy("customer_id").sum("price").show()


+-----------+----------+
|customer_id|sum(price)|
+-----------+----------+
|          1|     30000|
|          2|    -20000|
+-----------+----------+



In [0]:
customers.show()
sales.show()
cars.show()

+-----------+-----+---------+
|customer_id| name|     city|
+-----------+-----+---------+
|          1|John |Hyderabad|
|          2|Alice|  Chennai|
|          3| NULL|Bangalore|
+-----------+-----+---------+

+-------+-----------+------+----------+--------+
|sale_id|customer_id|car_id| sale_date|quantity|
+-------+-----------+------+----------+--------+
|      1|          1|   101|2024-01-01|       1|
|      2|          2|   102|2024-01-02|       2|
|      3|         99|   103|2024-01-03|       1|
+-------+-----------+------+----------+--------+

+------+-------+-----+------+
|car_id|  brand|model| price|
+------+-------+-----+------+
|   101| Toyota|Camry| 30000|
|   102|  Honda|Civic|-20000|
|   103|Hyundai|  i20| 15000|
+------+-------+-----+------+



In [0]:
customers.printSchema()
cars.printSchema()
sales.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- car_id: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: long (nullable = true)

root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- car_id: long (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: long (nullable = true)



In [0]:
df.show()

+------+-----------+-------+----------+--------+-----+---------+------+-----+------+
|car_id|customer_id|sale_id| sale_date|quantity| name|     city| brand|model| price|
+------+-----------+-------+----------+--------+-----+---------+------+-----+------+
|   101|          1|      1|2024-01-01|       1|John |Hyderabad|Toyota|Camry| 30000|
|   102|          2|      2|2024-01-02|       2|Alice|  Chennai| Honda|Civic|-20000|
+------+-----------+-------+----------+--------+-----+---------+------+-----+------+



In [0]:
customers.filter(col("name").isNull()).show()

+-----------+----+---------+
|customer_id|name|     city|
+-----------+----+---------+
|          3|NULL|Bangalore|
+-----------+----+---------+



In [0]:
cars.filter(col('price')<0).show()

+------+-----+-----+------+
|car_id|brand|model| price|
+------+-----+-----+------+
|   102|Honda|Civic|-20000|
+------+-----+-----+------+



In [0]:
sales.join(customers,'customer_id').show()

+-----------+-------+------+----------+--------+-----+---------+
|customer_id|sale_id|car_id| sale_date|quantity| name|     city|
+-----------+-------+------+----------+--------+-----+---------+
|          1|      1|   101|2024-01-01|       1|John |Hyderabad|
|          2|      2|   102|2024-01-02|       2|Alice|  Chennai|
+-----------+-------+------+----------+--------+-----+---------+



In [0]:
sales.join(customers,'customer_id','left_anti').show()

+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



In [0]:
customers.fillna({"name":"unknown"}).show()

+-----------+-------+---------+
|customer_id|   name|     city|
+-----------+-------+---------+
|          1|  John |Hyderabad|
|          2|  Alice|  Chennai|
|          3|unknown|Bangalore|
+-----------+-------+---------+



In [0]:
customers_cleaned=customers.withColumn("name", when(col("name").isNull(),"unknown").otherwise(trim(col("name")))).show()

+-----------+-------+---------+
|customer_id|   name|     city|
+-----------+-------+---------+
|          1|   John|Hyderabad|
|          2|  Alice|  Chennai|
|          3|unknown|Bangalore|
+-----------+-------+---------+



In [0]:
cars_cleaned=cars.filter(col('price')>=0).show()

+------+-------+-----+-----+
|car_id|  brand|model|price|
+------+-------+-----+-----+
|   101| Toyota|Camry|30000|
|   103|Hyundai|  i20|15000|
+------+-------+-----+-----+



In [0]:
customers_cleaned = customers.withColumn("name", when(col("name").isNull(),"unknown").otherwise(trim(col("name"))))
cars_cleaned = cars.filter(col('price')>=0)

df_cleaned = sales.join(customers_cleaned,'customer_id').join(cars_cleaned,'car_id')
display(df_cleaned)

car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000


In [0]:
valid_sales = sales \
    .join(customers_cleaned, "customer_id") \
    .join(cars_cleaned, "car_id")

### count before cleaning

In [0]:
total_sales = sales.count()

### count invalid rows

In [0]:
invalid_customers = sales.join(customers, "customer_id", "left_anti").count()
invalid_cars = sales.join(cars, "car_id", "left_anti").count()
negative_prices = cars.filter(col("price") < 0).count()

### final clean count

In [0]:
clean_count = valid_sales.count()

#### validation report

In [0]:
print("Total:", total_sales)
print("Invalid Customers:", invalid_customers)
print("Negative Prices:", negative_prices)
print("Final Clean:", clean_count)

Total: 3
Invalid Customers: 1
Negative Prices: 1
Final Clean: 1


### 1. Revenue per customer

In [0]:
revenue_df = valid_sales.withColumn(
    "revenue", col("price") * col("quantity")
)
revenue_df.show()
revenue_per_customer = revenue_df.groupBy("customer_id") \
    .agg(sum("revenue").alias("total_revenue"))
revenue_per_customer.show()

+------+-----------+-------+----------+--------+----+---------+------+-----+-----+-------+
|car_id|customer_id|sale_id| sale_date|quantity|name|     city| brand|model|price|revenue|
+------+-----------+-------+----------+--------+----+---------+------+-----+-----+-------+
|   101|          1|      1|2024-01-01|       1|John|Hyderabad|Toyota|Camry|30000|  30000|
+------+-----------+-------+----------+--------+----+---------+------+-----+-----+-------+

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|        30000|
+-----------+-------------+



### 2 brand wise sales

In [0]:
brand_sales = revenue_df.groupBy("brand") \
    .agg(sum("revenue").alias("brand_revenue"))
brand_sales.show()

+------+-------------+
| brand|brand_revenue|
+------+-------------+
|Toyota|        30000|
+------+-------------+



### 3 city-wise revenue

In [0]:
city_revenue = revenue_df.groupBy("city") \
    .agg(sum("revenue").alias("city_revenue"))
city_revenue.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Hyderabad|       30000|
+---------+------------+



In [0]:
revenue_df.createOrReplaceTempView("sales_data")

#### SQL ANALYSIS

### top 3 customers per city

In [0]:
%sql
select *
from (
    select *,
           row_number() over (partition by city order by revenue desc) as rn
    from sales_data
) t
where rn <= 3;

car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price,revenue,rn
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000,30000,1


### repeat customers

In [0]:
%sql
select customer_id, count(*) as purchase_count
from sales_data
group by customer_id
having count(*) > 1;

customer_id,purchase_count


### monthly trend

In [0]:
%sql
select month(sale_date) as month, sum(revenue) as total_revenue
from sales_data
group by month(sale_date)
order by month;

month,total_revenue
1,30000


In [0]:
revenue_per_customer.write.mode("overwrite").csv("/Volumes/workspace/phase5_schema/volume_phase5/car_sales_output")